# Payroll employees and pay (HMRC PAYE RTI) → ITL2, 2014–2026

Builds **payrolled employees and pay per ITL2 region per year** (annual averages of monthly data) and saves:
- `clean/itl2_2021/payroll_itl2.csv`: ITL2 **2021** (41 regions), taken **directly from the published NUTS2 tables** (NUTS2 2016 = ITL2 2021; same boundaries, `UK…` → `TL…` codes);
- `clean/payroll_itl2.csv`: ITL2 **2025** (46 regions), summed from the local-authority tables.

**Why:** an **administrative** source for employment and pay. It covers every PAYE employee (not a sample), so it doesn't suffer from falling survey response (unlike the APS/LFS used in `labour_market.ipynb`). See the admin-vs-survey review in the decisions log.

**Source:** ONS/HMRC *Earnings and employment from Pay As You Earn Real Time Information*, **seasonally adjusted** reference tables, accredited official statistics (`rtisasep2026.xlsx`, released 15 September 2026; raw copy in `raw/payroll/`). Monthly, **July 2014 – August 2026**.

| column | what | aggregation |
|---|---|---|
| `payrolled_employees` | people paid through PAYE in the month (daily average), **annual average** | count, sums across areas |
| `aggregate_pay_gbp_m` | total gross monthly pay, **annual average**, £ million | sums across areas |
| `mean_pay_monthly_gbp` | aggregate pay ÷ employees (derived from the counts) | recomputed at ITL2 |
| `median_pay_monthly_gbp` | ONS's published median, **average of the 12 monthly medians** | **not** aggregable; ITL2 2021 only, plus unchanged 2025 regions |
| `..._real_2025gbp` | pay in **2025 prices**, deflated by CPIH (ONS series L522, annual average) | |
| `n_months`, `full_year` | months of data in the year (2014: Jul–Dec; 2026: Jan–Aug) | |

**Key differences from APS / ASHE:**
- **Residence-based:** employees and pay are allocated to where the employee *lives* (HMRC address), like the APS labour counts and ASHE residence. It is **not** comparable with workplace measures (GVA, productivity, LBD job flows).
- **Employees only:** no self-employed. People, not jobs (someone with two PAYE jobs counts once).
- **Monthly pay, no hours:** pay differences mix hourly pay and hours (part-time share), unlike ASHE full-time hourly pay.
- Furloughed employees are included (2020–21); the latest months are early estimates and get revised.

Run top to bottom; only `BASE` needs editing.

In [1]:
import urllib.request
from pathlib import Path
import numpy as np
import pandas as pd

BASE   = Path("/Users/h.cantekin/Library/CloudStorage/OneDrive-LondonSchoolofEconomics/Desktop/regional-panel")
RAW    = BASE / "raw" / "payroll" / "rtisasep2026.xlsx"
CPIH   = BASE / "raw" / "payroll" / "cpih_l522.csv"
URL    = ("https://www.ons.gov.uk/file?uri=/employmentandlabourmarket/peopleinwork/earningsandworkinghours/datasets/"
          "realtimeinformationstatisticsreferencetableseasonallyadjusted/current/rtisasep2026.xlsx")
CPIH_URL = "https://www.ons.gov.uk/generator?format=csv&uri=/economy/inflationandpriceindices/timeseries/l522/mm23"
CROSSWALK = BASE / "lib" / "la_code_crosswalk.csv"
SPINE25, SPINE21 = BASE / "clean" / "geography_spine.csv", BASE / "clean" / "itl2_2021" / "geography_spine.csv"
LU25 = BASE / "raw" / "geography" / "LAD_(April_2025)_to_LAU1_to_ITL3_to_ITL2_to_ITL1_(January_2025)_Lookup_in_the_UK.csv"
LU21 = BASE / "raw" / "geography" / "LAD_(April_2023)_to_LAU1_to_ITL3_to_ITL2_to_ITL1_(January_2021)_Lookup_in_the_UK.csv"
REAL_YEAR = 2025

def fetch(url, path):
    """Download once; delete the file to refresh. (The 'current' RTI URL moves each month, so the workbook is kept in git.)"""
    if not path.exists():
        path.parent.mkdir(parents=True, exist_ok=True)
        req = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0"})
        path.write_bytes(urllib.request.urlopen(req, timeout=120).read())
fetch(URL, RAW); fetch(CPIH_URL, CPIH)

s25, s21 = pd.read_csv(SPINE25), pd.read_csv(SPINE21)
remap = dict(zip(*[pd.read_csv(CROSSWALK, dtype=str)[c] for c in ("old_code", "new_code")]))
print(pd.read_excel(RAW, sheet_name="Index", header=None).iloc[0, 0])

Earnings and employment from Pay As You Earn Real Time Information, UK: September 2026


## 1. Read the monthly tables

Each sheet has one column per area: codes on the row above the `Date` header row, names on the `Date` row, then one row per month. The trailing `UK` column and footnote rows are dropped.

In [2]:
MEASURES = {"Employees": "payrolled_employees", "Median pay": "median_pay_monthly_gbp", "Aggregate pay": "aggregate_pay_gbp"}

def read_sheet(prefix, level):
    name = next(s for s in pd.ExcelFile(RAW).sheet_names if s.endswith(f"{prefix} ({level})"))
    d = pd.read_excel(RAW, sheet_name=name, header=None)
    r = d.index[d[0].astype(str).str.strip() == "Date"][0]
    codes, names = d.iloc[r - 1, 1:], d.iloc[r, 1:]
    keep = codes.notna() & (names.astype(str) != "UK")
    body = d.iloc[r + 1:, [0] + list(keep[keep].index)]
    body.columns = ["month"] + codes[keep].astype(str).tolist()
    body["month"] = pd.to_datetime(body.month, format="%B %Y", errors="coerce")
    body = body.dropna(subset=["month"])
    out = body.melt(id_vars="month", var_name="code", value_name=MEASURES[prefix])
    out[MEASURES[prefix]] = pd.to_numeric(out[MEASURES[prefix]], errors="coerce")
    return out, dict(zip(codes[keep].astype(str), names[keep].astype(str)))

def read_level(level):
    frames = [read_sheet(p, level) for p in MEASURES]
    m = frames[0][0]
    for f, _ in frames[1:]:
        m = m.merge(f, on=["month", "code"], how="outer")
    return m, frames[0][1]

nuts2, nuts2_names = read_level("NUTS2")
lau, lau_names = read_level("LA")
for lab, d in [("NUTS2", nuts2), ("LA/LAU1", lau)]:
    print(f"{lab}: {d.code.nunique()} areas, {d.month.min():%b %Y} – {d.month.max():%b %Y}, "
          f"missing values: " + ", ".join(f"{c} {d[c].isna().sum()}" for c in MEASURES.values()))

NUTS2: 41 areas, Jul 2014 – Aug 2026, missing values: payrolled_employees 0, median_pay_monthly_gbp 0, aggregate_pay_gbp 41
LA/LAU1: 388 areas, Jul 2014 – Aug 2026, missing values: payrolled_employees 0, median_pay_monthly_gbp 0, aggregate_pay_gbp 388


## 2. Geography checks

1. **NUTS2 = ITL2 2021:** after `UK…` → `TL…`, the 41 codes must equal the 2021 spine's, and the names should match.
2. **LA tables → ITL2:** the "LA" tables are really **LAU1**: England, Wales and NI by local authority (including some **pre-reorganisation districts**, e.g. old Cumbria and North Yorkshire districts, which the crosswalk maps to their current authority and which are then summed), and **Scotland by 41 LAU1 sub-council areas** (codes `S30…`). LAU1s nest exactly in ITL2, so Scotland needs **no split rule**: e.g. *Arran and Cumbrae* and *North Ayrshire mainland* are published separately.
   - ITL2 2021: Scottish LAU1s via the 2021 lookup (all 41 match); other areas via crosswalk → 2021 spine.
   - ITL2 2025: Scottish LAU1s via the 2025 lookup; 11 Highland/Argyll LAU1s were recoded by 2025, and they map through their council (Highland and Argyll & Bute each sit wholly in one 2025 ITL2). Other areas via crosswalk → 2025 spine.
3. **Additivity:** LAU1 sums to ITL2 2021 are compared with the published NUTS2 values. If the two match, the 2025 sums can be trusted.

In [3]:
# 1. NUTS2 codes = ITL2 2021 codes
nuts2["itl2_code"] = "TL" + nuts2.code.str[2:]
assert set(nuts2.itl2_code) == set(s21.itl2_code), set(nuts2.itl2_code) ^ set(s21.itl2_code)
name21 = dict(zip(s21.itl2_code, s21.itl2_name))
diff = {f"TL{c[2:]}": (n, name21[f"TL{c[2:]}"]) for c, n in nuts2_names.items() if n.replace(",", "") != name21[f"TL{c[2:]}"].replace(",", "")}
print(f"NUTS2 -> ITL2 2021: all 41 codes match. Name differences (cosmetic): {diff if diff else 'none'}")

# 2. LAU1 -> ITL2 2021 and 2025
l21 = pd.read_csv(LU21, dtype=str); l25 = pd.read_csv(LU25, dtype=str, encoding="utf-8-sig")
lau_la = {c: (l21.set_index("LAU121CD").LAD23CD.get(c) if c.startswith("S30") else c) for c in lau.code.unique()}
lau_la = {c: remap.get(la, la) for c, la in lau_la.items()}                      # current LA code per LAU1 column
to21 = {c: (l21.set_index("LAU121CD").ITL221CD[c] if c.startswith("S30") else s21.set_index("la_code").itl2_code.get(lau_la[c]))
        for c in lau.code.unique()}
lau25_itl = l25.drop_duplicates("LAU125CD").set_index("LAU125CD").ITL225CD
to25 = {c: (lau25_itl[c] if c in lau25_itl.index else s25.set_index("la_code").itl2_code.get(lau_la[c])) for c in lau.code.unique()}
recoded = sorted(c for c in lau.code.unique() if c.startswith("S30") and c not in lau25_itl.index)
for lab, m, sp in [("2021", to21, s21), ("2025", to25, s25)]:
    miss = [c for c, v in m.items() if v is None or pd.isna(v)]
    assert not miss, f"unmapped areas ({lab}): {miss}"
    assert set(m.values()) == set(sp.itl2_code), f"regions without data ({lab}): {set(sp.itl2_code) - set(m.values())}"
    covered = {lau_la[c] for c in m if not c.startswith("S30")} | {lau_la[c] for c in m if c.startswith("S30")}
    assert set(sp.la_code) <= covered, f"spine LAs with no data ({lab}): {sorted(set(sp.la_code) - covered)}"
print(f"LAU1 columns: {lau.code.nunique()} -> every LA in both spines covered; "
      f"{len(recoded)} Scottish LAU1s recoded by 2025 mapped via their council: {[lau_names[c] for c in recoded]}")

# 3. additivity: LAU1 sums vs published NUTS2 (ITL2 2021)
SUM = ["payrolled_employees", "aggregate_pay_gbp"]
chk = (lau.assign(itl2_code=lau.code.map(to21)).groupby(["itl2_code", "month"])[SUM].sum(min_count=1)
          .join(nuts2.set_index(["itl2_code", "month"])[SUM], rsuffix="_pub"))
for c in SUM:
    gap = (100 * (chk[c] / chk[f"{c}_pub"] - 1)).abs()
    print(f"{c}: LAU1 sum vs published NUTS2, |gap| % - median {gap.median():.4f}, max {gap.max():.4f} "
          f"(worst: {gap.idxmax()[0]} {gap.idxmax()[1]:%b %Y})")

NUTS2 -> ITL2 2021: all 41 codes match. Name differences (cosmetic): {'TLG3': ('West Midlands (county)', 'West Midlands'), 'TLK1': ('Gloucestershire, Wiltshire and Bristol/Bath area', 'Gloucestershire, Wiltshire and Bath/Bristol area')}
LAU1 columns: 388 -> every LA in both spines covered; 11 Scottish LAU1s recoded by 2025 mapped via their council: ['Helensburgh and Lomond', 'Ross and Cromarty', 'Caithness and Sutherland', 'Inverness and Nairn', 'Badenoch and Strathspey', 'West Moray', 'North East Moray', 'Argyll and Bute Islands', 'Argyll and Bute Mainland', 'Lochaber', 'Skye and Lochalsh']
payrolled_employees: LAU1 sum vs published NUTS2, |gap| % - median 0.0001, max 0.0015 (worst: TLM6 Mar 2015)
aggregate_pay_gbp: LAU1 sum vs published NUTS2, |gap| % - median 0.0000, max 0.0000 (worst: TLM6 Jun 2015)


## 3. Annual averages and real pay

Per area and calendar year: average of the monthly **counts** (employees, aggregate pay); **mean pay = Σ aggregate pay ÷ Σ employees** over the months where both are published; **median pay = average of the monthly medians** (ONS publishes no annual median). 2014 (Jul–Dec) and 2026 (Jan–Aug) are partial years, flagged `full_year = False`. Seasonally adjusted data, so partial-year averages aren't biased by season, but they are still not full-year figures.

**Real pay (2025 £):** nominal × CPIH(2025) ÷ CPIH(year), with CPIH = ONS series **L522** (CPIH all items, 2015 = 100), annual average. CPIH is the ONS's lead inflation measure and the deflator it uses for real earnings. No real values for 2026 (no full-year CPIH yet).

In [4]:
cp = pd.read_csv(CPIH, header=None, names=["period", "value"], dtype=str)
cp = cp[cp.period.str.fullmatch(r"\d{4}")].astype({"period": int, "value": float}).set_index("period").value
print(f"CPIH (2015=100): 2014 {cp[2014]}, 2022 {cp[2022]}, {REAL_YEAR} {cp[REAL_YEAR]}; latest full year {cp.index.max()}")

def annual(d, key):
    d = d.assign(year=d.month.dt.year)
    both = d.dropna(subset=["payrolled_employees", "aggregate_pay_gbp"])
    g = d.groupby(key + ["year"]).agg(n_months=("month", "nunique"), payrolled_employees=("payrolled_employees", "mean"),
                                      aggregate_pay_gbp=("aggregate_pay_gbp", "mean"), median_pay_monthly_gbp=("median_pay_monthly_gbp", "mean"))
    g["mean_pay_monthly_gbp"] = (both.groupby(key + ["year"]).aggregate_pay_gbp.sum() / both.groupby(key + ["year"]).payrolled_employees.sum())
    return g.reset_index()

def finish(g):
    g = g.copy()
    g["full_year"] = g.n_months == 12
    g["aggregate_pay_gbp_m"] = g.pop("aggregate_pay_gbp") / 1e6
    defl = cp.get(REAL_YEAR) / g.year.map(cp)
    for c in ["mean_pay_monthly_gbp", "median_pay_monthly_gbp"]:
        g[c.replace("_gbp", f"_real_{REAL_YEAR}gbp")] = g[c] * defl
    g["payrolled_employees"] = g.payrolled_employees.round(0)
    cols = ["itl2_code", "itl2_name", "year", "n_months", "full_year", "payrolled_employees", "aggregate_pay_gbp_m",
            "mean_pay_monthly_gbp", "median_pay_monthly_gbp", f"mean_pay_monthly_real_{REAL_YEAR}gbp", f"median_pay_monthly_real_{REAL_YEAR}gbp"]
    return g[[c for c in cols if c in g] + [c for c in g if c not in cols]].sort_values(["itl2_code", "year"]).reset_index(drop=True)

# check: our derived annual mean pay vs averaging ONS's published monthly mean (NUTS2)
MEASURES["Mean pay"] = "pub_mean_pay"
pm = read_sheet("Mean pay", "NUTS2")[0]; del MEASURES["Mean pay"]
pm = pm.assign(itl2_code="TL" + pm.code.str[2:], year=pm.month.dt.year).groupby(["itl2_code", "year"]).pub_mean_pay.mean()
a21 = annual(nuts2, ["itl2_code"]).set_index(["itl2_code", "year"])
gap = 100 * (a21.mean_pay_monthly_gbp / pm - 1)
print(f"Annual mean pay: ours (Σpay/Σemployees) vs average of published monthly means: |gap| median {gap.abs().median():.3f}%, max {gap.abs().max():.3f}%")

CPIH (2015=100): 2014 99.6, 2022 120.5, 2025 138.0; latest full year 2025


Annual mean pay: ours (Σpay/Σemployees) vs average of published monthly means: |gap| median 0.004%, max 0.078%


## 4. ITL2 **2021** (41 regions): published NUTS2 values

Used as published: no aggregation, so the median is ONS's own regional median.

In [5]:
DIR21 = BASE / "clean" / "itl2_2021"
p21 = finish(annual(nuts2, ["itl2_code"]).assign(itl2_name=lambda d: d.itl2_code.map(name21)))
p21.to_csv(DIR21 / "payroll_itl2.csv", index=False)
print(f"Saved {DIR21 / 'payroll_itl2.csv'}: {p21.itl2_code.nunique()} regions x {p21.year.nunique()} years ({p21.year.min()}–{p21.year.max()}), {len(p21)} rows")
print(p21[p21.year == 2025].nlargest(3, "median_pay_monthly_gbp")[["itl2_name", "payrolled_employees", "median_pay_monthly_gbp", "mean_pay_monthly_gbp"]].round(0).to_string(index=False))

Saved /Users/h.cantekin/Library/CloudStorage/OneDrive-LondonSchoolofEconomics/Desktop/regional-panel/clean/itl2_2021/payroll_itl2.csv: 41 regions x 13 years (2014–2026), 533 rows
           itl2_name  payrolled_employees  median_pay_monthly_gbp  mean_pay_monthly_gbp
 Inner London - West             541938.0                  3693.0                8195.0
 Inner London - East            1233349.0                  3055.0                4650.0
Outer London - South             634451.0                  3019.0                4377.0


## 5. ITL2 **2025** (46 regions): summed from LAU1

Employees and aggregate pay are **summed** from the LAU1 tables (§2 showed the LAU1 sums reproduce the published NUTS2 totals), and mean pay is recomputed from the sums.

**Median pay:** medians can't be summed or averaged. A 2025 region whose LAs are **identical to one 2021 region's** takes that region's published NUTS2 median (`median_source = "published NUTS2"`). For the regions whose boundaries changed in 2025, the median is left blank. Use mean pay for those, or ASHE.

In [6]:
lau["itl2_code"] = lau.code.map(to25)
agg25 = lau.groupby(["itl2_code", "month"])[["payrolled_employees", "aggregate_pay_gbp"]].sum(min_count=1).reset_index()
p25 = annual(agg25.assign(median_pay_monthly_gbp=np.nan), ["itl2_code"])

# 2025 regions made of exactly the same LAU1 areas as a 2021 region -> published median
# (compared on the RTI areas themselves, so LAs split across regions, e.g. Argyll and Bute, are handled exactly)
inv = lambda m: {k: frozenset(c for c, v in m.items() if v == k) for k in set(m.values())}
sets21 = {v: k for k, v in inv(to21).items()}
same = {k: sets21.get(v) for k, v in inv(to25).items()}
same = {k: v for k, v in same.items() if v}
med = p21.set_index(["itl2_code", "year"]).median_pay_monthly_gbp
p25["median_pay_monthly_gbp"] = [med.get((same.get(c), y), np.nan) for c, y in zip(p25.itl2_code, p25.year)]
p25["median_source"] = np.where(p25.itl2_code.isin(same), "published NUTS2", "not available (boundary changed)")
p25["itl2_name"] = p25.itl2_code.map(dict(zip(s25.itl2_code, s25.itl2_name)))
p25 = finish(p25)
print(f"2025 regions identical to a 2021 region: {len(same)} of {s25.itl2_code.nunique()} (published median used); "
      f"changed: {sorted(set(s25.itl2_code) - set(same))}")

# sanity: the identical regions' summed employees equal the published 2021 values
cmp = p25[p25.itl2_code.isin(same)].assign(c21=lambda d: d.itl2_code.map(same)).merge(
    p21[["itl2_code", "year", "payrolled_employees"]], left_on=["c21", "year"], right_on=["itl2_code", "year"], suffixes=("", "_21"))
gap = (100 * (cmp.payrolled_employees / cmp.payrolled_employees_21 - 1)).abs().max()
print(f"Identical regions, summed vs published employees: max |gap| {gap:.4f}%"); assert gap < 0.01
assert np.isclose(p25.groupby("year").payrolled_employees.sum(), p21.groupby("year").payrolled_employees.sum(), rtol=1e-4).all()

p25.to_csv(BASE / "clean" / "payroll_itl2.csv", index=False)
print(f"Saved clean/payroll_itl2.csv: {p25.itl2_code.nunique()} regions x {p25.year.nunique()} years, {len(p25)} rows; UK totals match the 2021 file")

2025 regions identical to a 2021 region: 31 of 46 (published median used); changed: ['TLC3', 'TLC4', 'TLH4', 'TLH5', 'TLH6', 'TLK5', 'TLK6', 'TLK7', 'TLL3', 'TLL4', 'TLL5', 'TLM0', 'TLM1', 'TLM2', 'TLM3']
Identical regions, summed vs published employees: max |gap| 0.0005%
Saved clean/payroll_itl2.csv: 46 regions x 13 years, 598 rows; UK totals match the 2021 file


## 6. Notes

- **Use with:** `labour_counts_itl2.csv` (APS). Payrolled employees ÷ 16–64 population (`population_itl2.csv` / `age_itl2.csv`) gives an **admin employee rate**. It is not the same concept as the APS employment rate (no self-employed; includes people aged 65+).
- **Revisions:** the last two or three months of each release are early estimates. Rebuild with a later release (download the new workbook into `raw/payroll/` and update `RAW`) before final analysis; 2026 is a partial year.
- **Not for workplace comparisons:** residence-based, so it doesn't line up with GVA per hour or LBD job flows. For London commuter belts, residents' pay differs from the pay of jobs located there.